## Fields pulled from NIH RePORTER API

| Category | Fields |
|---|---|
| Temporal | fiscal_year, project_start_date, budget_start, budget_end, award_notice_date, is_new, award_type |
| Geographic | org_city, org_zipcode, cong_dist |
| Institutional | dept_type, funding_mechanism, direct_cost_amt, indirect_cost_amt, agency_ic_admin |
| Research Area | pref_terms, project_title, spending_categories_desc |
| Career Stage | activity_code (F=fellowship, K=career dev, R=research, T=training, P=program) |

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import requests
import pandas as pd
import time
import json

In [ ]:
# load Grant Witness terminations
df = pd.read_csv("/content/nih_v3_terminations.csv")

# Filter to NIH only known codes(drop the 5 unknown AX/DB codes)
NIH_CODES = {
    "GM","AI","AG","CA","MH","NS","HL","DK","HD","DA",
    "MD","DC","EY","ES","AR","HG","AA","OD","EB","DE",
    "NR","TW","LM","TR","AT"
}
df_nih = df[df["institute_code"].isin(NIH_CODES)].copy()
print(f"NIH grants total: {len(df_nih)}")

# get unique core award numbers
# core_award_number is the join key: e.g. "R01AA029044"
core_nums = df_nih["core_award_number"].dropna().unique().tolist()
print(f"Unique core project numbers: {len(core_nums)}")


In [ ]:

# batch query NIH Reporter API (this is a nightmare and will take 10 minutes
# but the API gets mad if you ask for more at a faster pace... :,(

def fetch_reporter_batch_v3(batch):
    wildcards = [f"{num}*" for num in batch]
    all_rows = []
    offset = 0

    while True:
        payload = {
            "criteria": {
                "project_nums": wildcards
            },
            "fields": [
                "project_num", "core_project_num", "fiscal_year",
                "org_name", "org_state", "org_city", "org_zipcode", "cong_dist",
                "award_amount", "direct_cost_amt", "indirect_cost_amt",
                "project_start_date", "project_end_date",
                "budget_start", "budget_end", "award_notice_date",
                "is_new", "award_type", "activity_code",
                "funding_mechanism", "organization_type", "dept_type",
                "agency_ic_admin", "spending_categories_desc",
                "pref_terms", "project_title", "is_active"
            ],
            "limit": 500,
            "offset": offset
        }
        r = requests.post(
            "https://api.reporter.nih.gov/v2/projects/search",
            json=payload
        )
        if r.status_code != 200:
            print(f"Error {r.status_code} at offset {offset}")
            break

        result = r.json()
        total = result['meta']['total']
        rows = result.get("results", [])
        all_rows.extend(rows)

        offset += len(rows)
        if offset >= total or not rows:
            break

        time.sleep(0.3)

    return all_rows

all_results = []
BATCH_SIZE = 25

for i in range(0, len(core_nums), BATCH_SIZE):
    batch = core_nums[i : i + BATCH_SIZE]
    results = fetch_reporter_batch_v3(batch)
    all_results.extend(results)
    print(f"Fetched batch {i//BATCH_SIZE + 1} running total: {len(all_results)} rows")
    time.sleep(0.5)  # API please be nice to me, do not change to faster if you query the api



In [ ]:
print(f"\ntotal reporter rows returned: {len(all_results)}")

# saved raw results as a checkpoint so I don't have to rerun evil API query
with open("/content/all_results_raw_v2.json", "w") as f:
    json.dump(all_results, f)
print("checkpoint saved...")

# Note: To reload later without re-fetching (skip the loop entirely next run):
# with open("/content/all_results_raw.json") as f:
#     all_results = json.load(f)

In [ ]:

# flatten nested fields
# organization fields are nested under g["organization"], flatten them out

def flatten_grant(g):
    org = g.pop("organization", {}) or {}
    g["org_name"]     = org.get("org_name")
    g["org_state"]    = org.get("org_state")
    g["org_city"]     = org.get("org_city")
    g["org_zipcode"]  = org.get("org_zipcode")
    g["cong_dist"]    = org.get("cong_dist")
    g["dept_type"]    = org.get("dept_type")
    return g

reporter_df = pd.DataFrame([flatten_grant(g) for g in all_results])
print(f"Reporter DataFrame shape: {reporter_df.shape}")
# after building reporter_df, add this diagnostic first
print("Null check:")
print(f"  core_project_num nulls: {reporter_df['core_project_num'].isna().sum()} / {len(reporter_df)}")
print(f"  project_num nulls:      {reporter_df['project_num'].isna().sum()} / {len(reporter_df)}")
print(reporter_df[['project_num', 'core_project_num']].head(5))

# calculate funding duration per core project
# reporter_df has one row per fiscal year per grant
# min/max fiscal year gives us the full funding span

funding_duration = (
    reporter_df
    .groupby("core_project_num")
    .agg(
        first_funded_year = ("fiscal_year", "min"),
        last_funded_year  = ("fiscal_year", "max"),
        years_funded      = ("fiscal_year", "nunique")
    )
    .reset_index()
)

print(funding_duration.describe())
print("\nLongest funded grants:")
print(funding_duration.sort_values("years_funded", ascending=False).head(10))




In [ ]:

# derive core project number from project_num (reliable field)
# example of "5R01AA029044-05" -> "R01AA029044"
reporter_df['core_derived'] = (
    reporter_df['project_num']
    .str.replace(r'^\d', '', regex=True)   # strip leading digit
    .str.split('-').str[0]                  # drop suffix after hyphen
)

print(f"\nSample derived core numbers:")
print(reporter_df[['project_num', 'core_derived']].head(5))


# Standard NIH core_project_num format very finicky: [letter][2 digits][2 letters][6 digits]
# Examples: R01AA029044, P01AG052350, T32AA007459
NIH_GRANT_PATTERN = r'^[A-Z][A-Z0-9]{1,2}[A-Z]{2}\d{6}$'

# check the split before moving forward
standard_mask = reporter_df['core_project_num'].str.match(NIH_GRANT_PATTERN, na=False)
print(f"Standard NIH grant rows:  {standard_mask.sum()} / {len(reporter_df)}")
print(f"Contract/non-standard:    {(~standard_mask).sum()} / {len(reporter_df)}")

# this should now be close to 5,616
reporter_grants_only = reporter_df[standard_mask].copy()
print(f"Unique core_project_num:  {reporter_grants_only['core_project_num'].nunique()}")

# dedup using core_project_num directly
reporter_latest = (
    reporter_grants_only
    .sort_values("fiscal_year", ascending=False)
    .drop_duplicates(subset="core_project_num", keep="first")
)
print(f"\nAfter dedup: {reporter_latest.shape}  ← should be close to 5,616")

# merge core_award_number on left, core_project_num on right
merged = df_nih.merge(
    reporter_latest,
    left_on="core_award_number",
    right_on="core_project_num",
    how="left"
)
merged = merged.merge(
    funding_duration,
    left_on="core_award_number",
    right_on="core_project_num",
    how="left",
    suffixes=("", "_duration")
)



print(f"\nMerged shape: {merged.shape}")
print(f"Grants with no Reporter match: {merged['core_project_num'].isna().sum()}")
print(f"Match rate: {merged['core_project_num'].notna().mean():.1%}")
# Quick summary of what got cut by funding age
print("\nFunding duration of terminated grants:")
print(merged["years_funded"].describe())
print(f"\nGrants funded 10+ years: {(merged['years_funded'] >= 10).sum()}")
print(f"Grants funded 15+ years: {(merged['years_funded'] >= 15).sum()}")
print(f"Grants funded 20+ years: {(merged['years_funded'] >= 20).sum()}")



In [ ]:
# save/download
merged.to_csv("/content/nih_merged.csv", index=False)
from google.colab import files
files.download('/content/nih_merged.csv')

In [ ]:
#who are the unmatched grants

unmatched = merged[merged['core_project_num'].isna()][['core_award_number', 'institute_code', 'title']]
print(f"Unmatched count: {len(unmatched)}")
print("\nBy institute:")
print(unmatched['institute_code'].value_counts())
print("\nSample unmatched:")
print(unmatched.head(10))